## tl;dr
This report reads sealed benchmark metrics only. It does not execute models, expose benchmark text, or select a winner without human review.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

run_dir = Path(os.environ['SIGNAL_SEMANTIC_LAB_RUN_DIR']).resolve()
report_path = run_dir / 'report-data.sanitized.json'
report = json.loads(report_path.read_text())
assert report['technical_benchmark_status'] in {'finalist_available', 'no_adoption'}
expected_safety = {
    'remote_reads': 0,
    'remote_read_scope': 'noisia_databases',
    'remote_writes': 0,
    'production_reads_writes': 0,
    'provider_calls': 0,
    'paid_jobs': 0,
    'serving_writes': 0,
    'new_open_source_model_artifact_download_bytes': 2267546218,
    'reused_open_source_model_artifact_bytes': 470642255,
    'pinned_open_source_model_artifact_bytes': 2738188473,
    'artifact_and_environment_download_upper_bound_bytes': 3488188473,
    'open_source_artifact_network_reads_only': True,
    'reused_sealed_staging_export': True,
}
assert report['safety'] == expected_safety
print({
    'technical_result': report['technical_result'],
    'denominator': report['export']['denominator'],
    'operator_review': report['human_review']['state'],
    'ready_for_10d': report['modeling_decision_ready_for_10d'],
    'report': str(report_path),
    'plan_digest': report['plan_digest'],
    'content_digest': report['export']['content_digest'],
})

## Context & Methods
The corrective preregistered matrix compares a locale-aware lexical reference and four fixed BERTopic profiles over two immutable multilingual embedding revisions. FASTopic was excluded before new results because its prior measured lower bound exceeded eight hours. Clustering, lexical representation and human naming are evaluated as separate layers.

### Key Assumptions
The normalized, rights-filtered canonical-root export is the benchmark denominator. Acquisition intent is retained only as provenance and is not semantic truth.

## Data

In [ ]:
pd.DataFrame([report['export']]).T.rename(columns={0: 'value'})

## Results

In [ ]:
result_stage = 'full' if 'full' in report['stages'] else 'calibration'
result_rows = []
for item in report['stages'][result_stage]['results']:
    metrics = item['metrics']
    result_rows.append({
        'candidate': item['candidate_key'],
        'seed': item['seed'],
        'coverage': metrics['coverage'],
        'c_npmi': metrics['c_npmi'],
        'diversity': metrics['topic_diversity'],
        'effective_topics': metrics['effective_topics'],
        'outlier_rate': metrics['outlier_rate'],
        'largest_cluster_share': metrics['cluster_size_distribution']['largest_cluster_share'],
        'stopword_topic_rate': metrics['representation']['majority_stopword_topic_rate'],
        'cluster_separation': metrics['cluster_geometry']['nearest_cluster_separation_mean'],
        'stability_input': metrics['denominator'],
        'duration_s': metrics['duration_seconds'],
        'peak_ram_gb': metrics['peak_rss_bytes'] / 1e9,
    })
results = pd.DataFrame(result_rows)
display(results.round(4))
rejections = pd.DataFrame(report['stages'][result_stage].get('resource_rejections', []))
if not rejections.empty:
    rejection_columns = [
        'candidate_key', 'state', 'reason',
        'projected_full_runtime_lower_bound_seconds',
    ]
    display(rejections[rejection_columns])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
quality = results.groupby('candidate')[['coverage', 'diversity']].mean()
quality.plot.bar(ax=axes[0], title=f'{result_stage}: coverage and diversity')
resources = results.groupby('candidate')[['duration_s', 'peak_ram_gb']].mean()
resources[['duration_s']].plot.bar(ax=axes[1], title=f'{result_stage}: discovery seconds')
resources[['peak_ram_gb']].plot.bar(ax=axes[2], title=f'{result_stage}: peak RAM (GB)')
plt.tight_layout()

## Validation

In [ ]:
observed = report['export']['exported']
excluded = sum(report['export']['excluded_by_reason'].values())
assert report['export']['denominator'] == observed + excluded
assert report['modeling_decision_ready_for_10d'] is False
expected_stability_input = report['stages'][result_stage]['record_count']
assert all(row['stability_input'] == expected_stability_input for row in result_rows)
review_frame = {'stable_operator_review_candidate': report['stable_operator_review_candidates']}
display(pd.DataFrame(review_frame))
pd.json_normalize(report['stability'], sep='.')

## Takeaways
Facts: technical runs, reconciliation and resource measurements above are sealed artifacts, and no probability is fabricated. Inference: stable technical finalists are eligible only for blinded operator review. Pending: no modeling decision or production runtime adoption exists until that review and a separate adoption ADR are complete; the single-scope corpus does not establish general multi-scope quality.